# RecurrentBitNet V2

Implementing the architecture described in `DESIGN.md`:

1. **Encoder Stack**: Non-recurrent parsing layers (first ~15%).
2. **Reasoning Core**: Complete 1-layer circuits repeated $R$ times with iteration embeddings and adaptive halting (ACT).
3. **Decoder Stack**: Non-recurrent output generation layers (last ~25%).
4. **BitLinear**: Ternary (1.58-bit) weight quantization.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, Any, List

# Set device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

## 1. BitLinear Implementation (1.58-bit)
We use the ternary quantization from `src.bitlinear` or implement a standalone version here for rapid prototyping.

In [ ]:
class BitLinear(nn.Linear):
    """
    BitLinear drop-in replacement for nn.Linear.
    Trains in FP32/BF16, quantizes weights to {-1, 0, 1} during forward pass.
    """
    def __init__(self, in_features: int, out_features: int, bias: bool = False, eps: float = 1e-5):
        super().__init__(in_features, out_features, bias)
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 1. Weight Quantization (absmax to ternary)
        gamma = self.weight.abs().mean().clamp(min=self.eps)
        w_scaled = self.weight / gamma
        w_quantized = torch.round(w_scaled).clamp(-1, 1)
        
        # STE (Straight Through Estimator) for gradients
        w_ternary = (w_quantized - self.weight).detach() + self.weight
        
        # 2. Activation Quantization (x scale)
        x_fp32 = x.float()
        norm = x_fp32.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        x_scaled = x_fp32 * norm
        
        # Quantize activations to 8-bit (-128 to 127) - Simplified for prototype
        x_quantized = torch.round(x_scaled * 127.0).clamp(-128, 127) / 127.0
        x_act = (x_quantized - x_scaled).detach() + x_scaled
        
        output = F.linear(x_act.to(x.dtype), w_ternary, self.bias)
        return output


## 2. Architecture Components
Encoder, Reasoning Step, and Decoder.

In [ ]:
@dataclass
class ModelConfig:
    d_model: int = 768
    n_heads: int = 12
    d_ff: int = 3072
    vocab_size: int = 32000
    max_seq_len: int = 2048
    
    encoder_blocks: int = 2
    reasoning_blocks: int = 4
    reasoning_recurrence: int = 3
    decoder_blocks: int = 2
    
    dropout: float = 0.1
    adaptive_halt: bool = True

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        norm = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return x * norm * self.weight

class BitFFN(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        # SwiGLU: d_model -> d_ff, then gated
        self.w1 = BitLinear(config.d_model, config.d_ff, bias=False)
        self.w2 = BitLinear(config.d_model, config.d_ff, bias=False)
        self.w3 = BitLinear(config.d_ff, config.d_model, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class BitAttention(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.d_model = config.d_model
        assert self.d_model % self.n_heads == 0
        self.head_dim = self.d_model // self.n_heads

        self.q_proj = BitLinear(config.d_model, config.d_model, bias=False)
        self.k_proj = BitLinear(config.d_model, config.d_model, bias=False)
        self.v_proj = BitLinear(config.d_model, config.d_model, bias=False)
        self.o_proj = BitLinear(config.d_model, config.d_model, bias=False)

    def forward(self, x, mask=None):
        B, L, D = x.size()
        
        q = self.q_proj(x).view(B, L, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, L, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, L, self.n_heads, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn = F.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, L, D)
        return self.o_proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.norm1 = RMSNorm(config.d_model)
        self.attn = BitAttention(config)
        self.norm2 = RMSNorm(config.d_model)
        self.ffn = BitFFN(config)

    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

## 3. Structural Components
Building `EncoderStack`, `ReasoningCore`, and `DecoderStack` as dictated by `DESIGN.md`.

In [ ]:
class EncoderStack(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.encoder_blocks)])

    def forward(self, x, mask=None):
        for block in self.blocks:
            x = block(x, mask)
        return x

class DecoderStack(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.decoder_blocks)])

    def forward(self, x, mask=None):
        for block in self.blocks:
            x = block(x, mask)
        return x

class ReasoningCore(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.reasoning_blocks)])
        # Iteration embeddings injected per pass to differentiate operations at each depth
        self.iteration_embeddings = nn.Parameter(torch.randn(config.reasoning_recurrence, config.d_model) * 0.02)
        
        # Optional: Adaptive Halting parameters
        if config.adaptive_halt:
            self.halt_scorer = nn.Sequential(
                nn.Linear(config.d_model, config.d_model // 2),
                nn.ReLU(),
                nn.Linear(config.d_model // 2, 1),
                nn.Sigmoid()
            )
            
    def forward(self, x, mask=None):
        outputs = []
        
        B, L, D = x.size()
        accumulated_halt_prob = torch.zeros(B, L, 1, device=x.device)
        
        for r in range(self.config.reasoning_recurrence):
            # Inject iteration embedding
            iter_emb = self.iteration_embeddings[r].view(1, 1, D)
            curr_x = x + iter_emb
            
            # Pass through reasoning blocks
            for block in self.blocks:
                curr_x = block(curr_x, mask)
            
            if self.config.adaptive_halt:
                halt_prob = self.halt_scorer(curr_x)
                accumulated_halt_prob = accumulated_halt_prob + halt_prob
                # ACT implementation simplifed for demonstration
            
            outputs.append(curr_x)
            x = curr_x
            
        return x, outputs

## 4. RecurrentBitNet V2 Model Assembly

In [ ]:
class RecurrentBitNetV2(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        
        self.encoder = EncoderStack(config)
        self.reasoning_core = ReasoningCore(config)
        self.decoder = DecoderStack(config)
        
        self.final_norm = RMSNorm(config.d_model)
        self.lm_head = BitLinear(config.d_model, config.vocab_size, bias=False)
        
        # Weight tying
        self.lm_head.weight = self.token_emb.weight
        
    def forward(self, idx, targets=None):
        B, L = idx.size()
        x = self.token_emb(idx)
        
        # Causal mask
        mask = torch.tril(torch.ones(L, L, device=x.device)).unsqueeze(0).unsqueeze(0)
        
        # 1. Encoding phase (parse input format)
        x = self.encoder(x, mask)
        
        # 2. Reasoning phase (recurrent cycles)
        x, iter_outputs = self.reasoning_core(x, mask)
        
        # 3. Decoding phase (generate output format)
        x = self.decoder(x, mask)
        
        x = self.final_norm(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            # Auxiliary loss calculation based on reasoning core outputs could be added here
            
        return logits, loss, iter_outputs

In [ ]:
# Run this cell to test the V2 Architecture Assembly
print("Testing Model Instantiation...")
config = ModelConfig()
model = RecurrentBitNetV2(config).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
print(f"Total Unique Parameters: {num_params:,}")

# Simulate dummy data
B, L = 2, 64
idx = torch.randint(0, config.vocab_size, (B, L)).to(DEVICE)
targets = torch.randint(0, config.vocab_size, (B, L)).to(DEVICE)

print("Testing Forward Pass...")
logits, loss, iter_outputs = model(idx, targets)

print(f"Logits shape: {logits.shape}")
print(f"Loss: {loss.item():.4f}")
print(f"Recurrence Steps Logged: {len(iter_outputs)}")
print("Test Passed successfully! The V2 parameters scale as expected without VRAM blowups.")

## 5. Training Infrastructure
Setting up the Differential Optimizer and a dummy dataset to test the progressive recurrence loop locally.

In [ ]:
# 5a. Dummy Dataset for Micro Testing
from torch.utils.data import DataLoader, TensorDataset

# Create 100 random batches of tokens
num_batches = 100
B, L = 4, 128
dummy_data = torch.randint(0, config.vocab_size, (num_batches * B, L)).to(DEVICE)
dummy_targets = torch.randint(0, config.vocab_size, (num_batches * B, L)).to(DEVICE)

dataset = TensorDataset(dummy_data, dummy_targets)
dataloader = DataLoader(dataset, batch_size=B, shuffle=True)

# 5b. Differential Optimizer
def create_optimizer(model, base_lr=1e-3):
    # As per DESIGN.md:
    # Encoder/Decoder: standard LR (base_lr)
    # Reasoning Core: higher LR (2 * base_lr)
    # Embeddings: lower LR (0.5 * base_lr)
    
    param_groups = {
        "encoder": [],
        "decoder": [],
        "reasoning": [],
        "embedding": [],
        "other": []
    }
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
            
        if "encoder" in name:
            param_groups["encoder"].append(param)
        elif "decoder" in name:
            param_groups["decoder"].append(param)
        elif "reasoning_core" in name:
            param_groups["reasoning"].append(param)
        elif "token_emb" in name or "lm_head" in name:
            param_groups["embedding"].append(param)
        else:
            param_groups["other"].append(param)
            
    optimizer = torch.optim.AdamW([
        {"params": param_groups["encoder"], "lr": base_lr},
        {"params": param_groups["decoder"], "lr": base_lr},
        {"params": param_groups["reasoning"], "lr": base_lr * 2.0},
        {"params": param_groups["embedding"], "lr": base_lr * 0.5},
        {"params": param_groups["other"], "lr": base_lr}
    ], weight_decay=0.1)
    
    return optimizer

optimizer = create_optimizer(model, base_lr=1e-3)
print("Optimizer created with differential learning rates.")

## 5c. FineWeb-Edu Streaming Setup (Optional Real Data)
Run this cell *instead* of 5a to use actual text data from `HuggingFaceFW/fineweb-edu`. Automatically tokenizes and yields standard PyTorch tensors.

In [ ]:
# 5c. Real Data Streaming
from datasets import load_dataset
from transformers import AutoTokenizer
import torch

# Load a standard 32k BPE tokenizer (Matches our ModelConfig vocab_size=32000)
# 'TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T' uses the standard 32000 Llama tokenizer without gating
tokenizer = AutoTokenizer.from_pretrained('TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Stream the FineWeb-Edu 10B token sample
fineweb_stream = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT', split='train', streaming=True)

seq_len = config.max_seq_len
batch_size = 4

def get_real_batch(stream_iter):
    inputs, targets = [], []
    while len(inputs) < batch_size:
        try:
            text = next(stream_iter)['text']
            tokens = tokenizer(text, truncation=True, max_length=seq_len + 1, return_tensors='pt')['input_ids'][0]
            
            # Need at least seq_len + 1 tokens to form an input/target pair.
            # If the text is shorter, we pad it for simplicity in this prototype.
            if len(tokens) < seq_len + 1:
                padding = torch.full((seq_len + 1 - len(tokens),), tokenizer.pad_token_id, dtype=torch.long)
                tokens = torch.cat([tokens, padding])
                
            inputs.append(tokens[:seq_len])
            targets.append(tokens[1:seq_len+1])
        except StopIteration:
            break
            
    if len(inputs) == 0:
        return None, None
        
    return torch.stack(inputs).to(DEVICE), torch.stack(targets).to(DEVICE)

# Example stream initialization
mapped_stream = iter(fineweb_stream)
real_idx, real_targets = get_real_batch(mapped_stream)
print(f"Stream yielded batch of shape {real_idx.shape} (inputs) and {real_targets.shape} (targets).")

## 6. Progressive Recurrence Training Loop
Executing the training loop that dynamically updates $R$ and calculates the auxiliary loss factor per recurrence pass.

In [ ]:
# 6. Progressive Training Loop
from tqdm.auto import tqdm

epochs = 1
alpha = 0.3 # Decay factor for auxiliary loss

# Progressive curriculum schedule: (step_threshold, recurrence_depth)
curriculum = [
    (0, 1),   # Start flat
    (20, 2),  # Shallow recurrence
    (50, 3)   # Deep recurrence
]

print("Starting Micro Training Run...")
global_step = 0
model.train()

loss_history = []
recurrence_history = []

for epoch in range(epochs):
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}")
    for batch_idx, (idx, targets) in enumerate(pbar):
        # 1. Update curriculum
        for threshold, R in reversed(curriculum):
            if global_step >= threshold:
                model.config.reasoning_recurrence = R
                break
                
        optimizer.zero_grad()
        
        # 2. Forward pass
        logits, base_loss, iter_outputs = model(idx, targets)
        
        # 3. Auxiliary Loss Calculation
        # L_tot = L_final + sum(alpha^(R-r) * L_step_r)
        aux_loss = 0.0
        R = model.config.reasoning_recurrence
        
        # Calculate loss for each intermediate step representation
        for r, hidden in enumerate(iter_outputs):
            # Normalization and tie weight projection
            step_normed = model.final_norm(hidden)
            step_logits = model.lm_head(step_normed)
            step_loss = F.cross_entropy(step_logits.view(-1, step_logits.size(-1)), targets.view(-1))
            
            # Apply decay weight (alpha^(R-r))
            decay_weight = alpha ** (R - (r + 1))
            aux_loss += decay_weight * step_loss
            
        total_loss = base_loss + aux_loss
        
        # 4. Backward Pass
        total_loss.backward()
        optimizer.step()
        
        # Track metrics
        global_step += 1
        loss_history.append(total_loss.item())
        recurrence_history.append(R)
        
        pbar.set_postfix({"Loss": f"{total_loss.item():.4f}", "R": R})

print("\nTraining Complete!")

## 7. Results & Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Plot Loss
ax1.plot(loss_history, label="Total Loss (Base + Aux)", color='blue')
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss Over Steps")
ax1.grid(alpha=0.3)

# Plot Recurrence Curriculum
ax2.step(range(len(recurrence_history)), recurrence_history, where='post', color='red', label="Recurrence Depth (R)")
ax2.set_ylabel("Recurrence (R)")
ax2.set_xlabel("Training Steps")
ax2.set_yticks([1, 2, 3])
ax2.set_title("Progressive Recurrence Depth Curriculum")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()